In [1]:
import csv
import random
import math


# ==========================================
# 1️⃣ Load Dataset
# ==========================================
FILE_PATH = "Mall_Customers.csv"
INCOME_COLUMN = "Annual Income (k$)"
SPENDING_COLUMN = "Spending Score (1-100)"

customer_data = []

with open(FILE_PATH, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        income_value = float(row[INCOME_COLUMN])
        spending_value = float(row[SPENDING_COLUMN])
        customer_data.append([income_value, spending_value])


# ==========================================
# 2️⃣ Standardization (Manual Z-Score)
# ==========================================
def calculate_mean_and_std(values):
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    std_dev = math.sqrt(variance)
    return mean_value, std_dev


income_list = [point[0] for point in customer_data]
spending_list = [point[1] for point in customer_data]

income_mean, income_std = calculate_mean_and_std(income_list)
spending_mean, spending_std = calculate_mean_and_std(spending_list)

scaled_customers = [
    [
        (point[0] - income_mean) / income_std,
        (point[1] - spending_mean) / spending_std
    ]
    for point in customer_data
]


# ==========================================
# 3️⃣ Manual K-Means
# ==========================================
NUM_CLUSTERS = 5
MAX_ITERATIONS = 1000
random.seed(42)

# Randomly initialize centroids
centroid_list = random.sample(scaled_customers, NUM_CLUSTERS)


def squared_manhattan_distance(point, centroid):
    return (
        abs(point[0] - centroid[0]) +
        abs(point[1] - centroid[1])
    )


for iteration in range(MAX_ITERATIONS):

    # Step 1: Create empty clusters dictionary
    clusters_dict = {cluster_id: [] for cluster_id in range(NUM_CLUSTERS)}

    # Step 2: Assign each point to nearest centroid
    for point in scaled_customers:
        distances = [
            squared_manhattan_distance(point, centroid)
            for centroid in centroid_list
        ]
        closest_cluster = distances.index(min(distances))
        clusters_dict[closest_cluster].append(point)

    # Step 3: Recompute centroids
    new_centroid_list = []

    for cluster_id in range(NUM_CLUSTERS):
        cluster_points = clusters_dict[cluster_id]

        if cluster_points:
            sorted_x = sorted(p[0] for p in cluster_points)
            sorted_y = sorted(p[1] for p in cluster_points)

            mid = len(cluster_points) // 2

            if len(cluster_points) % 2 == 0:
                median_x = (sorted_x[mid - 1] + sorted_x[mid]) / 2
                median_y = (sorted_y[mid - 1] + sorted_y[mid]) / 2
            else:
                median_x = sorted_x[mid]
                median_y = sorted_y[mid]

            new_centroid_list.append([median_x, median_y])
        else:
            # Reinitialize if cluster is empty
            new_centroid_list.append(random.choice(scaled_customers))

    # Step 4: Check convergence
    has_converged = True
    for i in range(NUM_CLUSTERS):
        movement = squared_manhattan_distance(
            centroid_list[i],
            new_centroid_list[i]
        )
        if movement > 1e-6:
            has_converged = False
            break

    centroid_list = new_centroid_list

    


# ==========================================
# 4️⃣ Final Cluster Assignment
# ==========================================
final_cluster_labels = []

for point in scaled_customers:
    distances = [
        squared_manhattan_distance(point, centroid)
        for centroid in centroid_list
    ]
    closest_cluster = distances.index(min(distances))
    final_cluster_labels.append(closest_cluster)


# ==========================================
# 5️⃣ Display Results (Original Scale)
# ==========================================
for cluster_id in range(NUM_CLUSTERS):

    original_points = [
        customer_data[index]
        for index, label in enumerate(final_cluster_labels)
        if label == cluster_id
    ]

    # Convert centroid back to original scale
    original_income = centroid_list[cluster_id][0] * income_std + income_mean
    original_spending = centroid_list[cluster_id][1] * spending_std + spending_mean

    print(f"\nCluster {cluster_id}")
    print("Size:", len(original_points))
    print("Centroid (Original Scale):",
          [round(original_income, 2), round(original_spending, 2)])
    print("Sample Points:", original_points[:10])


Cluster 0
Size: 33
Centroid (Original Scale): [74.0, 88.0]
Sample Points: [[18.0, 94.0], [19.0, 99.0], [23.0, 98.0], [28.0, 82.0], [29.0, 87.0], [33.0, 92.0], [33.0, 81.0], [38.0, 92.0], [69.0, 91.0], [70.0, 77.0]]

Cluster 1
Size: 33
Centroid (Original Scale): [23.0, 32.0]
Sample Points: [[15.0, 39.0], [15.0, 81.0], [16.0, 6.0], [16.0, 77.0], [17.0, 40.0], [17.0, 76.0], [18.0, 6.0], [19.0, 3.0], [19.0, 72.0], [19.0, 14.0]]

Cluster 2
Size: 37
Centroid (Original Scale): [79.0, 16.0]
Sample Points: [[70.0, 29.0], [71.0, 35.0], [71.0, 11.0], [71.0, 9.0], [72.0, 34.0], [73.0, 5.0], [73.0, 7.0], [74.0, 10.0], [75.0, 5.0], [76.0, 40.0]]

Cluster 3
Size: 15
Centroid (Original Scale): [99.0, 75.0]
Sample Points: [[85.0, 75.0], [87.0, 63.0], [87.0, 75.0], [88.0, 69.0], [97.0, 86.0], [98.0, 88.0], [99.0, 39.0], [99.0, 97.0], [101.0, 68.0], [103.0, 85.0]]

Cluster 4
Size: 82
Centroid (Original Scale): [54.0, 50.0]
Sample Points: [[30.0, 73.0], [34.0, 73.0], [37.0, 75.0], [39.0, 61.0], [39.0, 65